# structured-pruning on Colab / Kaggle

This notebook runs the [`structured-pruning`](https://github.com/thakerpragnesh/structured-pruning) repo on a free cloud GPU. It has four parts:

| Part | What it does | Needs a GPU? | How long |
|---|---|---|---|
| **1. Setup** (steps 1–3) | Detects Colab or Kaggle, clones the repo and installs it | no | minutes |
| **2. Verify** (steps 4–5) | Runs the two CUDA tests, then the full test suite | for step 4 | minutes |
| **3. Smoke runs** (step 6) | Runs the 8 experiment commands CI runs, plus `--tiny-check` on the real VGG16 and ResNet-18 classes | no | minutes |
| **4. Real run** (steps 7–12) | VGG16 / CIFAR-10 sweep comparing Max-3, L1, L2, random and K-Means. Every iteration is saved and the sweep resumes after a disconnect | **yes** | hours, set in step 7 |

**Before you press Run all**

* **Colab:** Runtime → Change runtime type → **T4 GPU**. Step 7 asks you to sign in to Google Drive, where the results are saved.
* **Kaggle:** in the right-hand panel set *Session options → Accelerator* to **GPU T4 x2** and turn *Internet* **on**. Internet needs a phone-verified account. For the long run, use *Save Version → Save & Run All (Commit)*: it runs in the background for up to 12 h, and `/kaggle/working/results` becomes the version's Output.
* **If the session dies,** Run all again. Finished methods are skipped. On Colab, the results are safe in Drive. On Kaggle, a new session starts with an empty `/kaggle/working`. First attach the previous version's output: *Add Input → Your Work → Notebooks → this notebook*. Step 7 then copies the run folder back.

**Not runnable here yet:** the full (non-`--smoke`) runs of experiments `02` (BERT/SST-2) and `03` (head redundancy) raise `NotImplementedError` in the repo itself, so this notebook runs only their smoke modes.

---
# Part 1 — Setup
## Step 1 — Platform and GPU

In [ ]:
import os, re, sys, shlex, subprocess, time
from pathlib import Path

if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    PLATFORM = "colab"
elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir():
    PLATFORM = "kaggle"
else:
    PLATFORM = "local"
WORK_DIR = {"colab": Path("/content"), "kaggle": Path("/kaggle/working")}.get(PLATFORM, Path.cwd())
print(f"platform: {PLATFORM}   python {sys.version.split()[0]}   work dir: {WORK_DIR}")

import torch

def _arch_supported(major, minor):
    """True if this torch build ships kernels the GPU can run (same major
    compute capability, minor <= the GPU's: an L4's sm_89 runs sm_86 code)."""
    for arch in torch.cuda.get_arch_list():
        m = re.fullmatch(r"sm_(\d+)\w*", arch)
        if m and int(m.group(1)) // 10 == major and int(m.group(1)) % 10 <= minor:
            return True
    return False

print(f"torch {torch.__version__}   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{major}{minor}), "
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
    if not _arch_supported(major, minor):
        print(f"WARNING: this torch build has no kernels for sm_{major}{minor} "
              f"({torch.cuda.get_arch_list()}) -- switch the accelerator to a T4.")
else:
    print("No GPU. Parts 1-3 still run (the CUDA test is skipped), but part 4 needs one:")
    print("  Colab: Runtime > Change runtime type > T4 GPU.   Kaggle: Accelerator > GPU T4 x2.")

## Step 2 — Get the code

This step clones `BRANCH` from GitHub. If you run the cell again, it pulls whatever has been pushed since. Colab and Kaggle see **only what is on GitHub**: push local commits first. The device fixes of 11 and 13 Oct and the `--device` flags have to be on GitHub, or step 4 fails. Run locally, the notebook uses the checkout it sits in.

In [ ]:
REPO_URL = "https://github.com/thakerpragnesh/structured-pruning.git"
BRANCH = "feature/generic-pruning-quantization"

def sh(cmd, cwd=None, check=True):
    """Run a shell command and stream its output into this cell (a bare
    subprocess.run() sends it to the kernel's log instead of the notebook)."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"exit code {rc}: {cmd}")
    return rc

def _local_checkout():
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "prunelib").is_dir() and (d / "pyproject.toml").is_file():
            return d
    return None

if PLATFORM == "local" and _local_checkout():
    REPO_DIR = _local_checkout()
    print(f"using the local checkout at {REPO_DIR}")
else:
    REPO_DIR = WORK_DIR / "structured-pruning"
    if not (REPO_DIR / ".git").is_dir():
        sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {shlex.quote(str(REPO_DIR))}")
    else:
        sh(f"git fetch --depth 1 origin {BRANCH} && git checkout -q FETCH_HEAD", cwd=REPO_DIR)

COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
sh("git log -1 --date=short --format='%h  %cd  %s'", cwd=REPO_DIR)

## Step 3 — Install

Colab and Kaggle already ship `torch`, `torchvision`, `transformers` and `pytest`. This step installs only the package itself, in editable mode, and leaves the preinstalled CUDA build of `torch` alone.

In [ ]:
PY = shlex.quote(sys.executable)
sh(f"{PY} -m pip install -q -e '.[dev,transformer-experiments,vision-experiments]'", cwd=REPO_DIR)

# New processes (the experiment scripts below) see the editable install, but
# this already-running kernel only reads .pth files at startup -- add the repo by hand.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import prunelib, torchvision
print(f"prunelib loaded from {Path(prunelib.__file__).parent}")
print(f"torch {torch.__version__}   torchvision {torchvision.__version__}")

---
# Part 2 — Verify
## Step 4 — The GPU tests

These two tests prune models on the GPU. One also runs with CUDA as torch's default device. They first passed on 2026-10-13, on a GTX 1650 (KT.md section 6). Here they check this runtime's GPU and drivers. On a GPU runtime both must report **PASSED**, not SKIPPED. A failure stops Run all here, before any GPU hours are spent.

In [ ]:
sh(f"{PY} -m pytest tests/test_graph.py -k cuda -v -rs", cwd=REPO_DIR)
if not torch.cuda.is_available():
    print("\n(skipped: no GPU on this runtime)")

## Step 5 — Full test suite

`-rs` lists the skip reason for every skipped test.

In [ ]:
sh(f"{PY} -m pytest tests/ -q -rs", cwd=REPO_DIR)

---
# Part 3 — Smoke and tiny-check runs
## Step 6

These are the commands CI runs on every push (`.github/workflows/tests.yml`), plus two `--tiny-check` runs. The tiny checks build the real torchvision VGG16 and ResNet-18 classes with random weights and no download. Every experiment takes `--device` and defaults to `cuda` when there is a GPU, so on a GPU runtime each command prints `device: cuda` and prunes on the GPU. `01 --tiny-check` builds a full-size VGG16 and fine-tunes it with Adam on the GPU, as part 4 does. If it runs out of memory here, part 4 will too.

The numbers these runs print are meaningless by design (random weights, fake data). What matters is that every command finishes without an error.

In [ ]:
SMOKE_RUNS = [
    "experiments/00_demo.py",
    "experiments/01_vgg_cifar10_sweep.py --smoke",
    "experiments/01_vgg_cifar10_sweep.py --tiny-check",  # real VGG16 class
    "experiments/02_bert_sst2_sweep.py --smoke",
    "experiments/03_head_redundancy.py --smoke",
    "experiments/04_coactivation.py",
    "experiments/05_ordering.py --budget 0.4",
    "experiments/06_generic_pruning.py --smoke",
    "experiments/06_generic_pruning.py --tiny-check",    # real ResNet-18 class
    "experiments/07_quantization.py",
]
for cmd in SMOKE_RUNS:
    print("\n" + "=" * 90)
    t0 = time.time()
    sh(f"{PY} {cmd}", cwd=REPO_DIR)
    print(f"[{time.time() - t0:.0f}s]")

---
# Part 4 — The real run: VGG16 on CIFAR-10

**Why not just `python experiments/01_vgg_cifar10_sweep.py`?** Running the script with no flag has two problems on Colab or Kaggle:

1. **It doesn't fit in a session.** It runs 5 methods × up to 15 iterations. Each iteration is a full epoch over all 50,000 CIFAR-10 images, upscaled to 224×224, at batch size 8. That is longer than one 12-hour Colab or Kaggle session. It also only prints, so a disconnect loses everything.
2. **Its baseline is an untrained head.** `build_vgg16` gives the ImageNet VGG16 a fresh, untrained 10-class head (`prunelib/vgg.py`). `run()` measures the baseline before any training, so the baseline is near chance (10%). Fine-tuning after each prune then *raises* accuracy, the "drop" goes negative, and the 1% stop rule can't fire in practice.

This part reuses the script's own building blocks: `build_vgg16`, `prune_vgg_layer`, and experiment 01's `_fine_tune` and `_evaluate`. It changes three things:

* **Baseline:** the new head is fine-tuned for `BASELINE_EPOCHS` before the baseline is measured. This happens once, and every method starts from the same checkpoint. Set `BASELINE_EPOCHS = 0` to measure the baseline the way `run()` does.
* **Budget:** the dataset size and batch size are settings, so a sweep fits in a session.
* **Saving and resuming:** each iteration is appended to `results.csv` as it finishes. Running again skips the methods that are already done.

## Step 7 — Settings

Each comment says what `run()` uses. The run folder's name is a hash of these settings (`METHODS` and the two `SAVE_*` flags excepted). Changing a setting starts a fresh run folder, and with the same settings you can split the methods across sessions, for example `["max_k", "l1"]` today and the rest tomorrow. Those sessions share one baseline and one `results.csv`.

In [ ]:
import hashlib, json, shutil

METHODS = ["max_k", "l1", "l2", "random", "kmeans"]  # run(): all five
PRUNE_STEP = 0.05          # run(): 0.05 -- every conv layer loses 5% of its current channels per iteration
ACC_DROP_THRESHOLD = 0.01  # run(): 0.01 -- a method stops once accuracy is >1 point below baseline
MAX_ITERATIONS = 15        # run(): 15
FINE_TUNE_EPOCHS = 1       # run(): 1, after every pruning iteration
LR = 1e-4                  # run(): 1e-4 (Adam, experiment 01's _fine_tune)
BASELINE_EPOCHS = 2        # run(): 0 -- the head is untrained when run() measures its baseline
TRAIN_SUBSET = 10_000      # run(): None, all 50,000 images
TEST_SUBSET = 2_000        # run(): None, all 10,000
BATCH_SIZE = 32            # run(): 8
IMAGE_SIZE = 224           # run(): 224
AMP = False                # run(): False -- fp16 mixed precision. Usually much faster on a T4 (tensor cores),
                           # but it changes the numerics; measured 4.4x slower on a GTX 1650.
NUM_WORKERS = 2
SEED = 0
DATASET = "cifar10"        # "fake": random-weight VGG16 + FakeData, as --tiny-check -- a quick wiring test
SAVE_FINAL_MODELS = False  # True: also save each method's final pruned model (a few hundred MB each)
SAVE_TO_DRIVE = True       # Colab only: results go to Google Drive, so they survive a disconnect

CONFIG = dict(dataset=DATASET, prune_step=PRUNE_STEP, acc_drop_threshold=ACC_DROP_THRESHOLD,
              max_iterations=MAX_ITERATIONS, fine_tune_epochs=FINE_TUNE_EPOCHS, lr=LR,
              baseline_epochs=BASELINE_EPOCHS, train_subset=TRAIN_SUBSET, test_subset=TEST_SUBSET,
              batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, amp=AMP, seed=SEED)
RUN_NAME = f"vgg16_{DATASET}_{hashlib.sha1(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:8]}"

if PLATFORM == "colab" and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive/structured-pruning-results")
elif PLATFORM == "kaggle":
    RESULTS_ROOT = Path("/kaggle/working/results")  # becomes the version's Output on Save Version
else:
    RESULTS_ROOT = REPO_DIR / "results"
# Kaggle saves all of /kaggle/working as output; keep the 170 MB download out of it.
DATA_DIR = {"colab": Path("/content/data"), "kaggle": Path("/tmp/data")}.get(PLATFORM, REPO_DIR / "data")
OUT_DIR = RESULTS_ROOT / RUN_NAME

def _copy_earlier_kaggle_output(out_dir, input_root=Path("/kaggle/input")):
    """A new Kaggle session starts with an empty /kaggle/working. If an earlier
    version's output is attached as an input, copy its run folder back so step 10 resumes."""
    if out_dir.exists():
        return
    for depth in range(3):
        for earlier in sorted(input_root.glob("*/" * (depth + 1) + f"results/{out_dir.name}")):
            shutil.copytree(earlier, out_dir)
            print(f"resuming from {earlier}")
            return

if PLATFORM == "kaggle":
    _copy_earlier_kaggle_output(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DATASET == "cifar10" and DEVICE == "cpu":
    raise RuntimeError("Part 4 with real CIFAR-10 needs a GPU runtime (see step 1). "
                       "DATASET = 'fake' runs the wiring test on CPU.")
if AMP and DEVICE == "cpu":
    raise RuntimeError("AMP = True needs a GPU runtime.")
print(f"run: {RUN_NAME}\nresults: {OUT_DIR}\ndata: {DATA_DIR}\ndevice: {DEVICE}")

## Step 8 — Data

The transforms are the same as `run_full()`'s (or `run_tiny_check()`'s, for `"fake"`). The subsets are fixed by `SEED`, so every method and every session sees the same images.

In [ ]:
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

if DATASET == "cifar10":
    transform = T.Compose([
        T.Resize(IMAGE_SIZE), T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_full = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=transform)
else:  # a wiring test, so a fixed handful of images whatever the subset settings say
    transform = T.Compose([T.Resize(IMAGE_SIZE), T.ToTensor()])
    train_full = torchvision.datasets.FakeData(size=64, image_size=(3, IMAGE_SIZE, IMAGE_SIZE),
                                               num_classes=10, transform=transform)
    test_full = torchvision.datasets.FakeData(size=32, image_size=(3, IMAGE_SIZE, IMAGE_SIZE),
                                              num_classes=10, transform=transform, random_offset=10_000)

def _fixed_subset(dataset, n, seed):
    if n is None or n >= len(dataset):
        return dataset
    idx = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(seed))[:n]
    return Subset(dataset, idx.tolist())

train_set = _fixed_subset(train_full, TRAIN_SUBSET, SEED)
test_set = _fixed_subset(test_full, TEST_SUBSET, SEED + 1)

def make_loaders(seed):
    """Seeded shuffling, so a method re-run after a disconnect sees the same batch order."""
    common = dict(num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda", persistent_workers=NUM_WORKERS > 0)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              generator=torch.Generator().manual_seed(seed), **common)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE * 2, **common)
    return train_loader, test_loader

print(f"train: {len(train_set):,} images   test: {len(test_set):,} images   "
      f"{-(-len(train_set) // BATCH_SIZE):,} steps per epoch")

## Step 9 — Baseline

The baseline is trained once and shared by every method. The first time, this downloads the ImageNet VGG16 weights (about 530 MB), fine-tunes for `BASELINE_EPOCHS` and saves a checkpoint to the run folder. Later sessions load that checkpoint instead. `acc_untrained_head` is the number `run()` would have used as its baseline.

In [ ]:
import importlib.util
from prunelib import build_vgg16, count_params, prune_vgg_layer
from prunelib.vgg import vgg_conv_bn_positions

# Experiment 01's own fine-tune and evaluate loops, so this run trains and
# scores exactly as the script does (a name starting with a digit can't be imported).
_spec = importlib.util.spec_from_file_location("exp01", REPO_DIR / "experiments" / "01_vgg_cifar10_sweep.py")
exp01 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(exp01)

def make_model(pretrained):
    return build_vgg16(pretrained=pretrained)

BASELINE_WEIGHTS = OUT_DIR / "baseline_state_dict.pt"
BASELINE_JSON = OUT_DIR / "baseline.json"  # written last, so its presence means the checkpoint is complete

if BASELINE_JSON.exists():
    baseline = json.loads(BASELINE_JSON.read_text())
    print(f"reusing the baseline in {OUT_DIR}")
else:
    torch.manual_seed(SEED)
    model = make_model(pretrained=DATASET == "cifar10").to(DEVICE)
    train_loader, test_loader = make_loaders(SEED)
    t0 = time.time()
    acc_untrained_head = exp01._evaluate(model, test_loader, DEVICE, amp=AMP)
    exp01._fine_tune(model, train_loader, DEVICE, epochs=BASELINE_EPOCHS, lr=LR, amp=AMP)
    baseline = {
        "acc_untrained_head": acc_untrained_head,
        "baseline_acc": exp01._evaluate(model, test_loader, DEVICE, amp=AMP),
        "params": count_params(model),
        "seconds": round(time.time() - t0, 1),
        "commit": COMMIT,
    }
    torch.save(model.state_dict(), BASELINE_WEIGHTS)
    BASELINE_JSON.write_text(json.dumps(baseline, indent=2))
    del model, train_loader, test_loader
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print(json.dumps(baseline, indent=2))

## Step 10 — Prune, fine-tune, evaluate

This step runs each method from the baseline. In every iteration it prunes `PRUNE_STEP` of every conv layer's channels with `prune_vgg_layer`, exactly as `run()` does. It then fine-tunes, evaluates and appends a row to `results.csv`. A method ends when its accuracy falls more than `ACC_DROP_THRESHOLD` below the baseline, or after `MAX_ITERATIONS`.

**After a disconnect,** Run all again. Finished methods are skipped, and an interrupted method restarts from the baseline. On Kaggle, attach the previous version's output first (see the top of the notebook). To estimate the whole sweep, look at the `sec` column of the first iteration: later iterations are faster, because the model shrinks.

In [ ]:
import csv, datetime

RESULTS_CSV = OUT_DIR / "results.csv"
FIELDS = ["method", "iteration", "params", "params_pruned_pct", "acc", "drop", "baseline_acc",
          "status", "seconds", "gpu", "commit", "finished_at"]
FINAL_STATUSES = {"stopped_at_threshold", "reached_max_iterations"}
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"

def _read_rows():
    if not RESULTS_CSV.exists():
        return []
    with open(RESULTS_CSV, newline="") as f:
        return list(csv.DictReader(f))

def _write_rows(rows, mode="w"):
    header = mode == "w" or not RESULTS_CSV.exists()
    with open(RESULTS_CSV, mode, newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if header:
            writer.writeheader()
        writer.writerows(rows)

finished = {r["method"] for r in _read_rows() if r["status"] in FINAL_STATUSES}
# A method an earlier session didn't finish restarts from the baseline, so drop its rows.
_write_rows([r for r in _read_rows() if r["method"] in finished])

for method in METHODS:
    if method in finished:
        print(f"{method}: finished in an earlier session, skipping")
        continue
    torch.manual_seed(SEED)
    model = make_model(pretrained=False)
    model.load_state_dict(torch.load(BASELINE_WEIGHTS, map_location="cpu"))
    model.to(DEVICE)
    train_loader, test_loader = make_loaders(SEED)
    n_layers = len(vgg_conv_bn_positions(model.features))

    print(f"\n=== {method}   baseline acc {baseline['baseline_acc']:.4f} ===")
    print(f"{'iter':>4} {'params':>12} {'pruned':>7} {'acc':>7} {'drop':>8} {'sec':>6}")
    for it in range(1, MAX_ITERATIONS + 1):
        t0 = time.time()
        for layer_position in range(n_layers):
            prune_vgg_layer(model, layer_position, PRUNE_STEP, method=method)
        exp01._fine_tune(model, train_loader, DEVICE, epochs=FINE_TUNE_EPOCHS, lr=LR, amp=AMP)
        acc = exp01._evaluate(model, test_loader, DEVICE, amp=AMP)
        drop = baseline["baseline_acc"] - acc
        params = count_params(model)
        pruned_pct = 100 * (1 - params / baseline["params"])
        status = ("stopped_at_threshold" if drop > ACC_DROP_THRESHOLD
                  else "reached_max_iterations" if it == MAX_ITERATIONS else "continued")
        seconds = time.time() - t0
        _write_rows([{
            "method": method, "iteration": it, "params": params, "params_pruned_pct": round(pruned_pct, 2),
            "acc": round(acc, 6), "drop": round(drop, 6), "baseline_acc": baseline["baseline_acc"],
            "status": status, "seconds": round(seconds, 1), "gpu": GPU_NAME, "commit": COMMIT,
            "finished_at": datetime.datetime.now().isoformat(timespec="seconds"),
        }], mode="a")
        print(f"{it:>4} {params:>12,} {pruned_pct:>6.1f}% {acc:>7.4f} {drop:>+8.4f} {seconds:>6.0f}")
        if status != "continued":
            print(f"  {method}: {status.replace('_', ' ')}")
            break

    if SAVE_FINAL_MODELS:  # the whole module: its shapes no longer match a fresh build_vgg16()
        torch.save(model, OUT_DIR / f"{method}_final_model.pt")
    del model, train_loader, test_loader
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print(f"\nall rows: {RESULTS_CSV}")

## Step 11 — Results

The first table shows accuracy per iteration. The second shows, for each method, the most pruning it reached while the accuracy drop stayed within the threshold. This is the paper's "params ↓ at <1% drop" figure for this run's budget. The chart gives each method its own panel, with the other methods in gray behind it.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_CSV)
methods = [m for m in METHODS if m in set(df["method"])] + sorted(set(df["method"]) - set(METHODS))
base = baseline["baseline_acc"]

print(f"test accuracy by iteration (baseline {base:.4f}):")
display(df.pivot(index="iteration", columns="method", values="acc").reindex(columns=methods).round(4))

within = df[df["drop"] <= ACC_DROP_THRESHOLD].sort_values("iteration").groupby("method").last()
print(f"\nmost pruning with accuracy drop <= {ACC_DROP_THRESHOLD:.0%} "
      f"(a method missing here exceeded it at iteration 1):")
display(within.reindex([m for m in methods if m in within.index])[["iteration", "params_pruned_pct", "acc", "drop"]])

SURFACE, INK, MUTED, GRID, AXIS, FOCUS = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7", "#2a78d6"
ncols = min(3, len(methods))
nrows = -(-len(methods) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.3 * nrows), sharex=True, sharey=True,
                         squeeze=False, facecolor=SURFACE)
def _curve(method):  # every curve starts from the unpruned baseline at 0%
    d = df[df["method"] == method]
    return [0.0, *d["params_pruned_pct"]], [base, *d["acc"]]

for ax, method in zip(axes.flat, methods):
    ax.set_facecolor(SURFACE)
    for other in methods:
        if other != method:
            ax.plot(*_curve(other), color=AXIS, linewidth=1, zorder=1)
    ax.plot(*_curve(method), color=FOCUS, linewidth=2, marker="o", markersize=5,
            markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=3)
    d = df[df["method"] == method]
    ax.axhline(base, color=MUTED, linewidth=1, linestyle="--", zorder=0)
    ax.axhline(base - ACC_DROP_THRESHOLD, color=MUTED, linewidth=1, linestyle=":", zorder=0)
    last = d.iloc[-1]
    ax.annotate(f"{last['acc']:.3f}", (last["params_pruned_pct"], last["acc"]), xytext=(5, 5),
                textcoords="offset points", color=INK, fontsize=9)
    ax.set_title(method, loc="left", color=INK, fontsize=11)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)
for idx in range(nrows * ncols):
    r, c = divmod(idx, ncols)
    if idx >= len(methods):  # an empty slot in the last row: label the panel above it instead
        axes[r, c].set_visible(False)
        axes[r - 1, c].tick_params(labelbottom=True)
        axes[r - 1, c].set_xlabel("parameters pruned (%)", color=MUTED)
    elif r == nrows - 1:
        axes[r, c].set_xlabel("parameters pruned (%)", color=MUTED)
for ax in axes[:, 0]:
    ax.set_ylabel("test accuracy", color=MUTED)
fig.suptitle(f"Accuracy vs. pruning, VGG16 / {'CIFAR-10' if DATASET == 'cifar10' else 'FakeData (wiring test)'}\n"
             "blue: the panel's method · gray: the other methods · dashed: baseline · dotted: stop line",
             x=0.01, ha="left", color=INK, fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "accuracy_vs_pruning.png", dpi=150, bbox_inches="tight", facecolor=SURFACE)
plt.show()

## Step 12 — Get the results out

* **Colab:** results are already in Drive, under `MyDrive/structured-pruning-results/<run>`. This cell also downloads a zip of the small files: the CSV, the JSON files and the chart.
* **Kaggle:** use *Save Version*. The run folder then appears under the version's *Output* tab.

In [ ]:
import zipfile

print(f"run folder: {OUT_DIR}")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:<32} {p.stat().st_size / 1e6:>8.1f} MB")

if PLATFORM == "colab":
    from google.colab import files
    archive = WORK_DIR / f"{RUN_NAME}.zip"
    with zipfile.ZipFile(archive, "w") as z:
        for p in OUT_DIR.iterdir():
            if p.suffix in {".csv", ".json", ".png"}:  # checkpoints stay in Drive
                z.write(p, arcname=f"{RUN_NAME}/{p.name}")
    files.download(str(archive))

---
## Troubleshooting

| Symptom | Fix |
|---|---|
| Step 1 says *No GPU* | Colab: Runtime → Change runtime type → T4 GPU. Kaggle: Accelerator → GPU T4 x2. Then Run all again. |
| `git clone` or `pip` fails on Kaggle | Internet is off: *Session options → Internet → On* (needs a phone-verified account). |
| Step 4 fails | Check that step 2 printed a commit dated 2026-10-13 or later, which includes both device fixes. If it did, this GPU or driver behaves differently from the GTX 1650 they were verified on: the failure output is what to fix. |
| `CUDA out of memory` in step 9 or 10 | Lower `BATCH_SIZE` in step 7. Note that this changes the run folder. |
| Colab disconnected mid-run | Reconnect, then Run all. Step 9 reuses the baseline, and step 10 skips finished methods. |
| Kaggle run cut off (12 h limit) | In a new session, attach the cut-off version's output (*Add Input → Your Work → Notebooks*). Step 7 should print `resuming from ...`. Then Save & Run All again. |
| You want the run exactly as `run()` does it | In step 7, set `BASELINE_EPOCHS = 0`, `TRAIN_SUBSET = TEST_SUBSET = None` and `BATCH_SIZE = 8`, then split `METHODS` across sessions. |